# 05 - Silver Future Calendar Rates

## Objetivo

Realizar a transformação e padronização da tabela
`future_calendar_rates`, proveniente da camada Bronze,
preparando os dados futuros de disponibilidade, preços e
ocupação projetada dos anúncios para análises do mercado
de Airbnb em João Pessoa - PB.

## Fonte

`airbnb_joao_pessoa.bronze.future_calendar_rates`

## Granularidade

Cada registro representa um anúncio (`listing_id`) em uma
determinada data (`date`).

Chave lógica:

`listing_id + date`

## Tabela de saída

`airbnb_joao_pessoa.silver.future_calendar_rates`

In [0]:
future_rates = spark.table("airbnb_joao_pessoa.bronze.future_calendar_rates")
display(future_rates)

In [0]:
print(f"Quantidade de registros: {future_rates.count()}")
print(f"Quantidade de colunas: {len(future_rates.columns)}")

In [0]:
# Nulos
from pyspark.sql.functions import sum, when, col

null_summary = (future_rates.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in future_rates.columns]))

display(null_summary)

In [0]:
# Duplicidade
duplicated_rates = (future_rates.groupBy("listing_id", "date").count().filter(col("count") > 1))

display(duplicated_rates)

In [0]:
# Período
from pyspark.sql.functions import min, max

display(future_rates.select(min("date").alias("min_date"), max("date").alias("max_date")))

In [0]:
# Datas que não são primeiro dia do mês
from pyspark.sql.functions import dayofmonth

display(future_rates.filter(dayofmonth("date") != 1).select("listing_id", "date"))

In [0]:
# Estatísticas
numeric_columns = ["vacant_days", "reserved_days", "occupancy", "revenue", "rate_avg", "booked_rate_avg", "booking_lead_time_avg", "length_of_stay_avg", "min_nights_avg", "native_booked_rate_avg", "native_rate_avg", "native_revenue"]

display(future_rates.select(numeric_columns).summary())

#### Validação da Future

In [0]:
# Dias
display(future_rates.filter((col("vacant_days") < 0) | (col("reserved_days") < 0)))

In [0]:
# Ocupação
display(future_rates.filter((col("occupancy") < 0) | (col("occupancy") > 1)))

In [0]:
# Valores monetários
display(
    future_rates.filter(
        (col("revenue") < 0) | (col("rate_avg") < 0) | (col("booked_rate_avg") < 0) | (col("native_booked_rate_avg") < 0) |
        (col("native_rate_avg") < 0) | (col("native_revenue") < 0))
)

In [0]:
# Métricas de estadia
display(
    future_rates.filter((col("booking_lead_time_avg") < 0) | (col("length_of_stay_avg") < 0) | (col("min_nights_avg") <= 0))
)

#### Indicadores derivados da Future

In [0]:
future_rates = (
    future_rates
    .withColumn("revenue_info_available", when(col("revenue").isNotNull(), True).otherwise(False))
    .withColumn("rate_info_available", when(col("rate_avg").isNotNull(), True).otherwise(False))
    .withColumn("occupancy_info_available", when(col("occupancy").isNotNull(), True).otherwise(False))
    .withColumn("availability_info_available", when(col("vacant_days").isNotNull() & col("reserved_days").isNotNull(),
            True).otherwise(False))
    .withColumn("stay_metrics_available", when(col("booking_lead_time_avg").isNotNull() &
            col("length_of_stay_avg").isNotNull() & col("min_nights_avg").isNotNull(), True).otherwise(False))
)

In [0]:
# Seleção final 
final_columns = ["listing_id", "date", "vacant_days", "reserved_days", "occupancy", "revenue", "rate_avg", "booked_rate_avg", "booking_lead_time_avg", "length_of_stay_avg", "min_nights_avg", "native_booked_rate_avg", "native_rate_avg", "native_revenue", "availability_info_available", "occupancy_info_available", "rate_info_available", "revenue_info_available", "stay_metrics_available"]

future_rates_silver = future_rates.select(final_columns)

display(future_rates_silver)

In [0]:
# Validação
print(f"Registros: {future_rates_silver.count()}")
print(f"Colunas: {len(future_rates_silver.columns)}")

In [0]:
display(future_rates_silver.groupBy("listing_id", "date").count().filter(col("count") > 1))